In [15]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

In [16]:
# 1. ARCHITECTURE LENET-5 MODIFIÉE

class LeNet5Modular(nn.Module):
    def __init__(self, num_classes=33, in_channels=1): 
        super(LeNet5Modular, self).__init__()
        
        # Couche C1: Convolution ( 1 x 32 x 32 to 6 x 28 x 28)
        self.c1 = nn.Conv2d(in_channels=in_channels, out_channels=6, kernel_size=5, stride=1)
        
        # Couche S2: Avg Pooling ( 6 x 28 x 28 to 6 x 14 x 14)
        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)
        
        # Couche C3: Convolution ( 6 x 14 x 14 to 16 x 10 x 10)
        self.c3 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1)
        
        # Couche S4: Avg Pooling ( 16 x 10 x 10 to 16 x 5 x 5)
        self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)
        
        # Couche C5: Fully Connected ( 16 * 5 * 5 = 400 to 120)
        self.c5 = nn.Linear(16 * 5 * 5, 120)
        
        # Couche F6: Fully Connected ( 120 to 84)
        self.f6 = nn.Linear(120, 84)
        
        # Couche de Sortie: (In: 84 to num_classes)
        self.output = nn.Linear(84, num_classes)
        
        self.tanh = nn.Tanh()

    def forward(self, x):
        # Activation Tanh classique de LeNet-5
        x = self.tanh(self.c1(x))
        x = self.s2(x)
        x = self.tanh(self.c3(x))
        x = self.s4(x)
        x = torch.flatten(x, 1) 
        x = self.tanh(self.c5(x))
        x = self.tanh(self.f6(x))
        x = self.output(x)
        return x

In [17]:
# CHARGEMENT DES DONNÉES (AMHCD)
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1), # Conversion RGB vers niveau de gris
    transforms.Resize((32, 32)), # Entrée 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


data_dir = './AMHCD_64' 

dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print(f"Nombre total d'images chargées : {len(dataset)}")
print(f"Nombre de classes détectées : {len(dataset.classes)}")

# 3. Division en jeux d'Entraînement (70%), Validation (15%) et Test (15%)
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(42)
)

# 4. Création des DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

Nombre total d'images chargées : 25740
Nombre de classes détectées : 1


In [18]:
# APPRENTISSAGE ET ÉVALUATION

def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=15):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        train_loss = running_loss / total
        train_acc = correct / total
        
        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
        
    return history

In [19]:
#  VISUALISATIONS (LOSS/ACC & MATRICE)

def plot_metrics(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Perte (Loss)
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_title('Courbes de Perte (Loss)')
    ax1.legend()
    
    # Précision (Accuracy)
    ax2.plot(history['train_acc'], label='Train Acc')
    ax2.plot(history['val_acc'], label='Val Acc')
    ax2.set_title('Courbes de Précision (Accuracy)')
    ax2.legend()
    
    plt.show()

def plot_confusion_matrix(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=False, cmap='Blues')
    plt.title('Matrice de Confusion (33 classes Tifinagh)')
    plt.xlabel('Prédictions')
    plt.ylabel('Vraies classes')
    plt.show()

In [20]:
def visualize_feature_maps(model, sample_image):
    model.eval()
    sample_image = sample_image.unsqueeze(0).to(device)
    
    # Extraction des sorties des couches convolutives
    c1_out = model.tanh(model.c1(sample_image))
    c3_out = model.tanh(model.c3(model.s2(c1_out)))
    
    # Visualisation de C1 (6 cartes)
    fig, axes = plt.subplots(1, 6, figsize=(15, 3))
    for i in range(6):
        axes[i].imshow(c1_out[0, i].cpu().detach().numpy(), cmap='gray')
        axes[i].axis('off')
    plt.suptitle("Cartes de caractéristiques - Couche C1")
    plt.show()